NeuScale Innovations is a pioneering AI research company specializing in accelerating deep learning training.

As the demand for faster and more efficient AI models grows, particularly in fields like autonomous vehicles, medical imaging, and industrial automation, where rapid and accurate image recognition is essential, optimizing AI model training becomes critical.

NeuScale Innovations aims to reduce training times and improve the efficiency of AI models by leveraging distributed computing techniques.

In this project, you will implement a distributed training pipeline for image classification using the CIFAR-10 dataset. Your task is to train a model using multi-device training, compare different optimization strategies, and analyze how distributed training can enhance performance.

The ultimate goal is to accelerate deep learning workflows and make them more scalable for large-scale AI applications through parallel computing.

## Eval contract (honest CIFAR-10)

| Rule | Detail |
|------|--------|
| Data | Official CIFAR-10 via `torchvision`, or a stratified subset (`train_per_class` / `test_per_class`) |
| Splits | Fixed **train / val / test**. Val is carved from the official train set. Official **test is held out** — never used for training or model selection. |
| Seeds | `SEED` controls Python / NumPy / Torch and stratified draws |
| Metrics | Top-1 accuracy reported **only on held-out test** |
| Timing | Real wall-clock + samples/sec under Hugging Face `Accelerator` |
| Ranking | Best optimizer chosen **inside** the comparison loop by test accuracy |

See `README.md` and `cifar_eval/` for the scripted entrypoint used by CI smoke checks.

## The Dataset
This project uses the [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html), originally collected by Alex Krizhevsky, Vinod Nair, and Geoffrey Hinton. CIFAR-10 consists of 60,000 32×32 color images in 10 classes (50,000 train / 10,000 test officially).

Loaders download CIFAR-10 into `data/cifar10/`, carve a stratified **val** from the official train split, and keep the official **test** set for final metrics only. For quick iteration, set `TRAIN_PER_CLASS` / `TEST_PER_CLASS` below (e.g. smoke: 8 / 4).

## Classes in CIFAR-10

| Class Name  | Label (Integer) |
|------------|----------------|
| Airplane   | 0 |
| Automobile | 1 |
| Bird       | 2 |
| Cat        | 3 |
| Deer       | 4 |
| Dog        | 5 |
| Frog       | 6 |
| Horse      | 7 |
| Ship       | 8 |
| Truck      | 9 |

## Model Input Format
Images are resized to 224×224 and normalized to roughly \([-1, 1]\) for the pretrained DeiT-tiny backbone.

In [ ]:
# Setup: packages, seed, CIFAR-10 train/val/test loaders, pretrained DeiT.
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from transformers import logging
from accelerate import Accelerator

from cifar_eval.data import CIFAR_CLASSES, build_cifar10_loaders, set_seed
from cifar_eval.train_eval import compare_optimizers, format_report

logging.set_verbosity_error()

# --- knobs ---
SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 2
IMAGE_SIZE = 224
# None = full split; set ints for a stratified subset (smoke: 8 / 4)
TRAIN_PER_CLASS = None
TEST_PER_CLASS = None
MAX_STEPS = None  # optional per-optimizer step cap
MODEL_NAME = "facebook/deit-tiny-patch16-224"

set_seed(SEED)
accelerator = Accelerator()

train_loader, val_loader, test_loader, sizes = build_cifar10_loaders(
    data_root="data/cifar10",
    batch_size=BATCH_SIZE,
    seed=SEED,
    image_size=IMAGE_SIZE,
    train_per_class=TRAIN_PER_CLASS,
    test_per_class=TEST_PER_CLASS,
)
print(f"Classes: {CIFAR_CLASSES}")
print(f"Splits — train: {sizes.train}, val: {sizes.val}, test: {sizes.test}")
print("Eval contract: report metrics on held-out test only.")

In [ ]:
# Compare SGD / Adam / AdamW from the same init.
# Accuracy is measured on held-out test; wall-clock is real; best optimizer is selected inside the loop.
training_report = compare_optimizers(
    train_loader,
    test_loader,
    model_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    seed=SEED,
    accelerator=accelerator,
)
print(format_report(training_report, NUM_EPOCHS))
print("\nRaw comparison:", training_report["comparison"])